In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Project 3(Advanced Multi-Source Data Pipeline)\2.1_product_data.csv")
df.head()

,Title,Price,Rating,Availability
0,A Light in the Attic,Â£51.77,Three,In stock
1,Tipping the Velvet,Â£53.74,One,In stock
2,Soumission,Â£50.10,One,In stock
3,Sharp Objects,Â£47.82,Four,In stock
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock


In [3]:
# converting invalid type to float
# Price format > Remove £ symbol and convert to float
print(df.dtypes)

Title           str
Price           str
Rating          str
Availability    str
dtype: object


In [4]:
df["Price"] = (
    df["Price"]
    .astype(str)
    .str.replace("Â£", "", regex=False)
    .str.strip()
)

df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

print(df["Price"].head())
print(df["Price"].dtype)

0    51.77
1    53.74
2    50.10
3    47.82
4    54.23
Name: Price, dtype: float64
float64


In [5]:
# Rating text Convert "One", "Two"... → numerical (1–5)
df["Rating"] = df["Rating"].replace({
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
})
print(df["Rating"].head())

0    3
1    1
2    1
3    4
4    5
Name: Rating, dtype: object


In [6]:
# Availability text  - Extract number of books in stock

df["Stock"] = df["Availability"].str.extract(r"(\d+)").astype(float)

df["Stock"] = df["Availability"].replace({
    "In stock": 1,
    "Out of stock": 0
})
print(df["Stock"])

0     1
1     1
2     1
3     1
4     1
5     1
6     1
7     1
8     1
9     1
10    1
11    1
12    1
13    1
14    1
15    1
16    1
17    1
18    1
19    1
Name: Stock, dtype: object


In [7]:
# Extra spaces / special chars : Clean title and text columns
df["Title"] = df["Title"].str.strip()
print(df["Title"])

0                                  A Light in the Attic
1                                    Tipping the Velvet
2                                            Soumission
3                                         Sharp Objects
4                 Sapiens: A Brief History of Humankind
5                                       The Requiem Red
6     The Dirty Little Secrets of Getting Your Dream...
7     The Coming Woman: A Novel Based on the Life of...
8     The Boys in the Boat: Nine Americans and Their...
9                                       The Black Maria
10       Starving Hearts (Triangular Trade Trilogy, #1)
11                                Shakespeare's Sonnets
12                                          Set Me Free
13    Scott Pilgrim's Precious Little Life (Scott Pi...
14                            Rip it Up and Start Again
15    Our Band Could Be Your Life: Scenes from the A...
16                                                 Olio
17    Mesaerion: The Best Science Fiction Storie

In [8]:
# Duplicates : Removing duplicate book titles
# Check duplicate rows
print(df.duplicated().sum())

# Remove duplicate rows
df = df.drop_duplicates()

0


In [9]:
# Data type conversion: Ensures correct numerical types
print(df.dtypes)

Title               str
Price           float64
Rating           object
Availability        str
Stock            object
dtype: object


In [10]:
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df["Stock"] = pd.to_numeric(df["Stock"], errors="coerce")

print(df["Rating"].dtype)
print(df["Stock"].dtype)

int64
int64


In [11]:
# Feature Engineering : Price range categories, Rating groups, Stock status

# price range categories
df["Price_Range"] = pd.cut(
    df["Price"],
    bins=[0, 20, 40, 60, float("inf")],
    labels=["Low", "Medium", "High", "Very High"]
)

# rating groups
df["Rating_Group"] = pd.cut(
    df["Rating"],
    bins=[0, 2, 3, 4, 5],
    labels=["Poor", "Average", "Good", "Excellent"]
)

# Stock status
df["Stock_Status"] = df["Availability"].replace({
    "In stock": "Available",
    "Out of stock": "Not Available"
})

In [12]:
print(df.head())

                                   Title  Price  Rating Availability  Stock  \
0                   A Light in the Attic  51.77       3     In stock      1   
1                     Tipping the Velvet  53.74       1     In stock      1   
2                             Soumission  50.10       1     In stock      1   
3                          Sharp Objects  47.82       4     In stock      1   
4  Sapiens: A Brief History of Humankind  54.23       5     In stock      1   

  Price_Range Rating_Group Stock_Status  
0        High      Average    Available  
1        High         Poor    Available  
2        High         Poor    Available  
3        High         Good    Available  
4        High    Excellent    Available  


In [13]:
df.to_csv("cleaned_product_data.csv", index=False)